# ACO for CVRP — Experiment Analysis

Comparative analysis of Greedy, ACO, EAS, and MMAS on CVRPLIB A-series instances.

In [1]:
%matplotlib inline
import importlib
import json
import pathlib
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
import src.viz, src.vrp, src.metrics, src.solution
importlib.reload(src.viz)
importlib.reload(src.vrp)
importlib.reload(src.metrics)
importlib.reload(src.solution)

from src.viz import plot_routes, plot_convergence, plot_boxplots, plot_scaling
from src.vrp import load_vrp, load_sol
from src.metrics import iterations_to_delta, stagnation_length

RESULTS_DIR  = pathlib.Path("../results/main")
SMAX_DIR_32  = pathlib.Path("../results/smax_32")
SMAX_DIR_45  = pathlib.Path("../results/smax_45")
DATA_DIR     = pathlib.Path("../data")

df = pd.read_csv(RESULTS_DIR / "results.csv")
df["feasible"] = df["feasible"].astype(bool)
df["relative_error"] = pd.to_numeric(df["relative_error"], errors="coerce")
df["s_max"] = pd.to_numeric(df["s_max"], errors="coerce")

main = df[df["feasible"] & df["s_max"].isna()]
print(f"Loaded {len(df)} total runs, {len(main)} feasible unconstrained")

FileNotFoundError: [Errno 2] No such file or directory: '..\\results\\main\\results.csv'

## H3.1 — Solution Quality

Compare mean relative error vs best-known across algorithms. Box plots show distribution across seeds.

In [ ]:
print("=== Mean relative error vs best known (lower is better) ===")
display(
    main.groupby("algorithm")["relative_error"]
    .agg(["mean", "std", "min"])
    .rename(columns={"mean": "Mean RE", "std": "Std RE", "min": "Best RE"})
    .round(4)
    .sort_values("Mean RE")
)

print("\n=== Mean total distance per instance ===")
display(
    main.groupby(["algorithm", "instance"])["total_dist"]
    .mean()
    .round(1)
    .unstack("algorithm")
)

plot_boxplots(main)

## Route Comparison — Optimal vs. Best Found

Side-by-side comparison of known optimal routes (from `.sol` files) with the best routes found by each algorithm (seed 42, 200 iterations).

In [ ]:
import src.greedy, src.aco_base, src.eas, src.mmas
importlib.reload(src.greedy)
importlib.reload(src.aco_base)
importlib.reload(src.eas)
importlib.reload(src.mmas)

from src.greedy import solve_greedy
from src.aco_base import ACOBase
from src.eas import EAS
from src.mmas import MMAS
from src.solution import compute_total_dist

INSTANCES = [
    ("data/A/A-n32-k5.vrp", 5),
    ("data/A/A-n37-k6.vrp", 6),
    ("data/A/A-n45-k6.vrp", 6),
    ("data/A/A-n55-k9.vrp", 9),
    ("data/A/A-n60-k9.vrp", 9),

    ("data/A/A-n80-k10.vrp", 10),
]
SEED = 42

def run_solver(algo_name, inst, nv):
    random.seed(SEED); np.random.seed(SEED)
    if algo_name == "greedy":
        sol = solve_greedy(inst, nv)
    elif algo_name == "aco":
        sol = ACOBase(n_ants=20, n_iterations=200).solve(inst, nv)[0]
    elif algo_name == "eas":
        sol = EAS(n_ants=20, n_iterations=200, e=20).solve(inst, nv)[0]
    elif algo_name == "mmas":
        sol = MMAS(n_ants=20, n_iterations=200, rho=0.2).solve(inst, nv)[0]
    return sol.routes, sol.total_dist

for vrp_path, n_vehicles in INSTANCES:
    sol_path = vrp_path.replace(".vrp", ".sol")
    inst = load_vrp(f"../{vrp_path}")

    optimal_routes, optimal_known_cost = load_sol(f"../{sol_path}")
    optimal_cost = compute_total_dist(optimal_routes, inst.dist, inst.depot)

    results = {}
    for algo_name in ["greedy", "aco", "eas", "mmas"]:
        routes, cost = run_solver(algo_name, inst, n_vehicles)
        results[algo_name] = (routes, cost)

    # Plot: optimal + each algorithm side by side
    n_cols = 1 + len(results)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))

    plot_routes(optimal_routes, inst.coords,
                f"Optimal ({optimal_known_cost:.0f})", ax=axes[0])

    for i, (algo_name, (routes, cost)) in enumerate(results.items(), start=1):
        re = (cost - optimal_cost) / optimal_cost * 100
        plot_routes(routes, inst.coords,
                    f"{algo_name} ({cost:.0f}, +{re:.1f}%)", ax=axes[i])

    fig.suptitle(f"{inst.name}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## H3.2 — Scalability

Relative error and runtime vs instance size (number of customers).

In [ ]:
plot_scaling(main)

elapsed = (
    main.groupby(["algorithm", "n_customers"])["elapsed_sec"]
    .mean()
    .reset_index()
)
fig, ax = plt.subplots(figsize=(10, 5))
for algo in elapsed["algorithm"].unique():
    g = elapsed[elapsed["algorithm"] == algo].sort_values("n_customers")
    ax.plot(g["n_customers"], g["elapsed_sec"], "-o", label=algo)
ax.set_xlabel("Customers")
ax.set_ylabel("Mean elapsed (s)")
ax.set_yscale("log")
ax.set_title("Runtime scaling (log scale)")
ax.legend()

## H3.3 — Convergence

Iterations to 20% improvement, stagnation length, and convergence curves.

In [ ]:
records = []
for jf in RESULTS_DIR.glob("*.json"):
    run = json.loads(jf.read_text())
    if not run.get("convergence") or run["algorithm"] == "greedy":
        continue
    records.append({
        "algorithm":    run["algorithm"],
        "instance":     run["instance"],
        "seed":         run["seed"],
        "iter_to_20pct": iterations_to_delta(run["convergence"], delta=0.20),
        "stagnation":   stagnation_length(run["convergence"]),
    })

conv_df = pd.DataFrame(records)
display(
    conv_df.groupby(["algorithm", "instance"])[["iter_to_20pct", "stagnation"]]
    .mean()
    .round(1)
)

target = "A-n32-k5"
curves = {}
for jf in RESULTS_DIR.glob("*.json"):
    run = json.loads(jf.read_text())
    if run["instance"] == target and run["seed"] == 42 and run.get("convergence"):
        curves[run["algorithm"]] = run["convergence"]

plot_convergence(curves, f"Convergence — {target} (seed 42)")

## H3.4 — S_max and α/β Sensitivity

How tight distance constraints shift the optimal α/β parametrisation.

In [ ]:
sw = pd.concat([
    pd.read_csv(SMAX_DIR_32 / "results.csv"),
    pd.read_csv(SMAX_DIR_45 / "results.csv"),
], ignore_index=True)
sw["feasible"] = sw["feasible"].astype(bool)
sw["relative_error"] = pd.to_numeric(sw["relative_error"], errors="coerce")
sw["s_max"] = pd.to_numeric(sw["s_max"], errors="coerce")

# --- Unconstrained: rank by relative error (best-known is available) ---
unc = sw[sw["s_max"].isna() & sw["feasible"]]
if not unc.empty:
    best_unc = (
        unc.groupby(["algorithm", "param_alpha", "param_beta"])["relative_error"]
        .agg(["mean", "std", "min", "count"])
        .round(4)
        .sort_values("mean")
    )
    print("=== Unconstrained — all α/β combos by mean relative error ===")
    display(best_unc)

# --- Constrained: rank by total_dist (relative_error is N/A) ---
con = sw[sw["s_max"].notna() & sw["feasible"]]
if not con.empty:
    top_overall = (
        con.groupby(["algorithm", "param_alpha", "param_beta"])["total_dist"]
        .agg(["mean", "std", "min", "count"])
        .round(1)
        .sort_values("mean")
    )
    print("\n=== Constrained — all α/β combos by mean total distance ===")
    display(top_overall)

    print("\n=== Constrained — mean total distance per instance × α/β ===")
    display(
        con.groupby(["algorithm", "instance", "param_alpha", "param_beta"])["total_dist"]
        .mean()
        .round(1)
        .unstack(["param_alpha", "param_beta"])
    )
else:
    print("Constrained: no feasible runs")

# --- Feasibility rate ---
print("\n=== Feasibility rate ===")
constraint_label = sw["s_max"].isna().map({True: "unconstrained", False: "constrained"})
display(
    sw.groupby(["algorithm", constraint_label])["feasible"]
    .mean()
    .round(3)
    .unstack()
)

# --- Per α/β feasibility (constrained only) ---
con_all = sw[sw["s_max"].notna()]
if not con_all.empty:
    print("\n=== Constrained — feasibility rate per α/β ===")
    display(
        con_all.groupby(["algorithm", "param_alpha", "param_beta"])["feasible"]
        .mean()
        .round(3)
        .unstack("param_beta")
    )

# --- Dead-end ratio and success rate from JSON ---
sr_records = []
for smax_dir in [SMAX_DIR_32, SMAX_DIR_45]:
    for jf in smax_dir.glob("*.json"):
        run = json.loads(jf.read_text())
        if not run.get("dead_end_ratio"):
            continue
        sr_records.append({
            "algorithm":         run["algorithm"],
            "instance":          run["instance"],
            "s_max":             run.get("s_max"),
            "alpha":             run.get("params", {}).get("alpha"),
            "beta":              run.get("params", {}).get("beta"),
            "mean_dead_ends":    sum(run["dead_end_ratio"]) / len(run["dead_end_ratio"]),
            "mean_success_rate": sum(run["success_rate"]) / len(run["success_rate"]),
        })
sr_df = pd.DataFrame(sr_records)

print("\n=== Dead-ends & success rate by algorithm × constraint ===")
sr_constraint = sr_df["s_max"].isna().map({True: "unconstrained", False: "constrained"})
display(
    sr_df.groupby(["algorithm", sr_constraint])
    [["mean_dead_ends", "mean_success_rate"]]
    .mean()
    .round(3)
)

# --- Per α/β breakdown for constrained runs ---
sr_con = sr_df[sr_df["s_max"].notna()]
if not sr_con.empty:
    print("\n=== Constrained — dead-ends & success rate per α/β ===")
    display(
        sr_con.groupby(["algorithm", "alpha", "beta"])
        [["mean_dead_ends", "mean_success_rate"]]
        .mean()
        .round(3)
    )

### Route visualisation — unconstrained vs. constrained (S_max = 300)

How tight distance constraints force shorter, tighter vehicle loops.

In [ ]:
SMAX_INSTANCES = [
    ("data/A/A-n32-k5.vrp", 5),
    ("data/A/A-n45-k6.vrp", 6),
]
SEED = 42
SMAX_VAL = 300

for vrp_path, n_vehicles in SMAX_INSTANCES:
    inst = load_vrp(f"../{vrp_path}")

    algos = ["aco", "mmas"]
    fig, axes = plt.subplots(2, len(algos), figsize=(6 * len(algos), 12))

    for col, algo_name in enumerate(algos):
        for row, (s_max, label) in enumerate([(float("inf"), "Unconstrained"), (SMAX_VAL, f"S_max = {SMAX_VAL}")]):
            random.seed(SEED); np.random.seed(SEED)
            if algo_name == "aco":
                sol = ACOBase(n_ants=20, n_iterations=200).solve(inst, n_vehicles, s_max=s_max)[0]
            else:
                sol = MMAS(n_ants=20, n_iterations=200, rho=0.2).solve(inst, n_vehicles, s_max=s_max)[0]

            ax = axes[row, col]
            plot_routes(sol.routes, inst.coords,
                        f"{algo_name.upper()} — {label}\n({sol.total_dist:.0f}, {len(sol.routes)} routes)",
                        ax=ax)

    fig.suptitle(f"{inst.name} — Unconstrained vs. Constrained", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()